In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    rename,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    drop_duplicate_columns,
    common_translate,
    collapse_col,
    split_data,
    fix_redundancies,
    SpenderID,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`ET` and {term}`DSO` data is not connected (see [](general:ic)). The following table lists the different types of rows, whhich occur in this file and which ID combination they use.

In [ ]:
data = split_data(data, ["donor_et_dso", "donor_et_id_et"])
assert len(data) == 2, "Not 4 different row types present!?"

In [ ]:
et = data["donor_et_id_et"].reset_index()
assert not et.duplicated().any()
dso = data["donor_et_dso"].reset_index()

The following summary shows the {term}`ET` data. This data is not in a longitudinal format and there is one row for each donor.

In [ ]:
display_data_doc(data=et)

The following summary shows the {term}`DSO` data. This data on the other hand is in the longitudinal format.

In [ ]:
display_data_doc(data=dso)

In the {term}`DSO` data the rows with `diagnosis_type="Todesursache"` also only occur once, therefore they can also be treated as not longitudinal data. We used these rows to construct a table with donor death information similar to the {term}`ET` data. Because the death reasons are hard to consolidate the data is redudant, we drop the {term}`ET` data.

In [ ]:
assert (
    dso.groupby("donor_et_dso")["diagnosis_type"]
    .value_counts()
    .loc[(slice(None), "Todesursache")]
    .max()
    == 1
), "Multiple deaths by DSO?"

dso_death = drop_col_few_distinct(
    dso[dso["diagnosis_type"] == "Todesursache"], verbose=False
).drop(columns="diagnosis_type")
dso_nondeath = drop_col_few_distinct(
    dso[dso["diagnosis_type"] != "Todesursache"], verbose=False
)
assert (dso_death["communication_date"] == dso_death["death_date"]).all()
dso_death.drop(columns="communication_date", inplace=True)
death = pd.merge(
    et,
    dso_death,
    left_on="donor_et_id_et",
    right_on="donor_et_dso",
    how="outer",
    validate="1:1",
).drop(columns="donor_et_dso")
death = rename(
    death,
    {
        "donor_et_id_et": "donor_et_id_et",
        "death_reason_et": "death_reason_et",
        "death_reason_icd_et": "death_reason_icd_et",
        "death_reason_icd_code_et": "death_reason_icd_code_et",
        "death_date_et": "death_date_et",
        "diagnosis": "death_reason_dso",
        "diagnosis_code": "death_reason_icd_code_dso",
        "diagnosis_code_additional": "death_reason_icd_code_additional_dso",
        "disease_start_date": "death_disease_start_date",
        "disease_end_date": "death_disease_end_date",
        "brain_damage_type": "brain_damage_type_death",
        "brain_damage_localisation": "brain_damage_localisation_death",
        "death_date": "death_date_dso",
        "death_type": "death_type_final_diagnosis",
    },
)
display_data_doc(data=death)

As the other rows of the {term}`DSO` are in a longitudinal format, we merge the death data rows to each longitudinal row.

In [ ]:
data = pd.merge(
    death,
    dso_nondeath,
    left_on="donor_et_id_et",
    right_on="donor_et_dso",
    how="outer",
    validate="1:m",
)

In [ ]:
display_data_doc(data=data)

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

No further row filtering was necessary (see [](general:rf)).

### Unit Conversions

We applied common translations (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

### Consolidating Columns

Because we only include {term}`DSO` data, no consolidation besides the donor identifier was necessary (see [](general:crc)).

In [ ]:
# you can manually add if necessary
# nur das, der rest passt nicht
red = {
    "donor_et_id_et": ["donor_et_id_et", "donor_et_dso"],
    "death_date": ["death_date_et", "death_date_dso"],
}
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `disease_start_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["disease_start_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
y = ["yes"]
icd = r"[A-Z]\d\d(:?\.\d?\d)?"


class DonorPostmortemDiagnoses(SpenderID):
    brain_damage_localisation_death: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Brain Damage Location (For the death diagnosis)",
        description="Where, if any, was brain damage localized?",
        isin=["supratentoriell", "gesamtcerebral", "infratentoriell"],
    )
    brain_damage_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Brain Damage Type (For this diagnosis)",
        description="What kind, if any, of brain damage was diagnosed?",
        isin=["atraumatisch - primär", "traumatisch - primär", "sekundär"],
    )
    brain_damage_type_death: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Brain Damage Type (For the Death diagnosis)",
        description="What kind, if any, of brain damage was diagnosed for the death diagnosis?",
        isin=["atraumatisch - primär", "traumatisch - primär", "sekundär"],
    )
    colon_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Colon Exclusion",
        description="Was the colon excluded as a possible donation by this diagnosis?",
        isin=y,
    )
    communication_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communication Date",
        description="When was the diagnosis communicated?",
    )
    death_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Date",
        description="When was the patient declared dead?",
    )
    death_disease_end_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="End Date for Fatal Disease",
        description="When was the end date of the disease that caused the patient death?",
    )
    death_disease_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Start Date for Fatal Disease",
        description="When was the start date of the disease that caused the patient death?",
    )
    death_reason_et: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Reason (by ET)",
        description="What was the death reason (text) for this patient as reported by ET?",
    )
    death_reason_icd_code_additional_dso: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Reason Additional Code (by DSO)",
        description="Additional ICD code for the death reason of this patient as reported by DSO?",
        str_matches=icd,
    )
    death_reason_icd_code_dso: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Reason Code (by DSO)",
        description="ICD code for the death reason of this patient as reported by DSO?",
        str_matches=icd,
    )
    death_reason_icd_code_et: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Reason Code (by ET)",
        description="ICD code for the death reason of this patient as reported by ET?",
        str_matches=icd,
    )
    death_reason_icd_et: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death reason code text (by ET)",
        description="What was the death reason (ICD text) for this patient as reported by ET",
    )
    death_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Type",
        description="Was it a natural death? (For this diagnosis)",
        isin=["natürlich", "nicht natürlich", "ungeklärt"],
    )
    death_type_final_diagnosis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Type",
        description="Was it a natural death? (For the death diagnosis)",
        isin=["natürlich", "nicht natürlich", "ungeklärt"],
    )
    diagnosis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diagnosis",
        description="Text of this diagnosis",
    )
    diagnosis_additional: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Quality of Diagnosis",
        description="Additional information on the diagnosis",
        isin=[
            "Gesichert",
            "Verdacht auf",
            "Zustand nach",
            "Zur Beobachtung",
            "Histologisch gesichert",
            "Sonstiges",
        ],
    )
    diagnosis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diagnosis Code",
        description="ICD code of this diagnosis",
    )
    diagnosis_code_additional: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Additional Diagnosis Code",
        description="Additional ICD code of this diagnosis",
        str_matches=icd,
    )
    diagnosis_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diagnosis Type",
        description="What type of diagnosis is this?",
        isin=[
            "Aufnahmediagnose",
            "Begleiterkrankung",
            "Vorerkrankung",
            "zur Hirnschädigung führende Diagnose",
            "Komplikation",
            "Kontraindikation",
            "Risikofaktor",
            "Differentialdiagnose",
            "Tumor",
        ],
    )
    disease_end_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="End Date for this disease",
        description="When was the end date of the disease?",
    )
    disease_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Start Date For this disease",
        description="When was the start date of this disease?",
    )
    heart_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Heart Exclusion",
        description="Was the heart excluded as a possible donation by this diagnosis",
        isin=y,
    )
    kidney_left_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Left Kidney Exclusion",
        description="Was the left kidney excluded as a possible donation by this diagnosis",
        isin=y,
    )
    kidney_right_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Right Kidney Exclusion",
        description="Was the right kidney excluded as a possible donation by this diagnosis",
        isin=y,
    )
    liver_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Liver Exclusion",
        description="Was the liver excluded as a possible donation by this diagnosis",
        isin=y,
    )
    lung_left_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Left Lung Exclusion",
        description="Was the left lung excluded as a possible donation by this diagnosis",
        isin=y,
    )
    lung_right_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Right Lung Exclusion",
        description="Was the right lung excluded as a possible donation by this diagnosis",
        isin=y,
    )
    pancreas_excluded: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pancreas Exclusion",
        description="Was the pancreas excluded as a possible donation by this diagnosis",
        isin=y,
    )
    treated: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Treated",
        description="Was this disease treated",
        isin=y + ["no"],
    )

    class Config:
        title = "Donor Postmortem Diagnosis Dataset"
        description = "Each row represents a diagnosis relevant for the death of the donor. The data is based on the 'element_spender_postmortem_diagnosen.csv' file. It contains data from the ET and DSO."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemDiagnoses, data)

In [ ]:
DonorPostmortemDiagnoses.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemDiagnoses.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)